# Explore Cerner sandbox FHIR data

Run cells top to bottom. Part 1 needs no login. Part 2 uses your app secret.

In [1]:
import os, json, requests

TENANT = "ec2458f2-1e24-41c8-b71b-0e701af7583d"  # Cerner public sandbox tenant
HEADERS = {"Accept": "application/fhir+json"}

def show(data):
    print(json.dumps(data, indent=2)[:3000])  # first 3000 chars is enough to see the shape

## Part 1 - Open sandbox (no auth, fake patients)

In [2]:
OPEN_URL = f"https://fhir-open.cerner.com/r4/{TENANT}"

r = requests.get(f"{OPEN_URL}/Patient", params={"family": "SMART", "_count": 5}, headers=HEADERS)
bundle = r.json()
show(bundle)

{
  "resourceType": "Bundle",
  "id": "5aa95931-b2d3-4155-97c8-7539a7221a13",
  "type": "searchset",
  "link": [
    {
      "relation": "self",
      "url": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Patient?family=SMART&_count=5"
    },
    {
      "relation": "next",
      "url": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Patient?-pageContext=c3fed8cf-112a-4414-b74d-9ab52d5ec1bb&-pageDirection=NEXT"
    }
  ],
  "entry": [
    {
      "fullUrl": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Patient/12742399",
      "resource": {
        "resourceType": "Patient",
        "id": "12742399",
        "meta": {
          "versionId": "210394",
          "lastUpdated": "2026-07-28T14:51:15.000Z"
        },
        "text": {
          "status": "generated",
          "div": "<div xmlns=\"http://www.w3.org/1999/xhtml\"><p><b>Patient</b></p><p><b>Name</b>: SMART II, SANDY</p><p><b>Status</b>: Active</p><p><b>DOB</b

In [3]:
# Pick the first patient and look at the data a transfer summary would use
patient_id = bundle["entry"][0]["resource"]["id"]
print("Patient:", patient_id)

for resource in ["Condition", "Encounter", "Observation", "MedicationRequest", "AllergyIntolerance"]:
    r = requests.get(f"{OPEN_URL}/{resource}", params={"patient": patient_id, "_count": 3}, headers=HEADERS)
    print(f"\n===== {resource} ({r.status_code}) =====")
    show(r.json())

Patient: 12742399

===== Condition (504) =====
{
  "message": "Gateway Timeout",
  "code": 504
}

===== Encounter (200) =====
{
  "resourceType": "Bundle",
  "id": "3d626f0f-67f7-4ca2-aa07-6a9217c4fa31",
  "type": "searchset",
  "link": [
    {
      "relation": "self",
      "url": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Encounter?patient=12742399&_count=3"
    },
    {
      "relation": "next",
      "url": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Encounter?patient=12742399&-pageContext=b2122c69-c57e-43a3-b5fa-7bfc384e64b0&-pageDirection=NEXT"
    }
  ],
  "entry": [
    {
      "fullUrl": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Encounter/97955594",
      "resource": {
        "resourceType": "Encounter",
        "id": "97955594",
        "meta": {
          "versionId": "0",
          "lastUpdated": "2020-11-05T18:49:47.000Z"
        },
        "text": {
          "status": "generated",
        

In [ ]:
# Bed info lives in Location resources. Cerner requires a filter: -physicalType=bd means "bed"
r = requests.get(f"{OPEN_URL}/Location", params={"-physicalType": "bd", "_count": 5}, headers=HEADERS)
show(r.json())

## Readable version

Every FHIR record has a `text.div` field: a short human-readable summary written by Cerner. Stripping the HTML gives one line per record, which is a good input for the AI summary.

In [ ]:
import re

def readable(resource):
    html = resource.get("text", {}).get("div", "")
    return re.sub(r"<[^>]+>", " ", html).split()  # strip HTML tags

for resource in ["Patient", "Encounter", "Condition", "Observation", "MedicationRequest", "AllergyIntolerance"]:
    params = {"_id": patient_id} if resource == "Patient" else {"patient": patient_id, "_count": 10}
    r = requests.get(f"{OPEN_URL}/{resource}", params=params, headers=HEADERS)
    print(f"\n===== {resource} ({r.status_code}) =====")
    for entry in r.json().get("entry", []):
        print("-", " ".join(readable(entry["resource"])))

## Part 2 - Your app (DischargeAgent) with the secret

The next cell pops up a box asking for the secret. Paste it there; it is not saved in the notebook.

In [ ]:
CLIENT_ID = "92e1d486-4f2d-4218-81b8-cbeb9d3748f8"  # DischargeAgent_2 (System app) Client ID
from getpass import getpass
SECRET = getpass("Paste Cerner secret: ")

TOKEN_URL = f"https://authorization.cerner.com/tenants/{TENANT}/protocols/oauth2/profiles/smart-v1/token"
SECURE_URL = f"https://fhir-ehr-code.cerner.com/r4/{TENANT}"

r = requests.post(
    TOKEN_URL,
    auth=(CLIENT_ID, SECRET),
    data={"grant_type": "client_credentials", "scope": "system/Patient.read system/Encounter.read system/Condition.read"},
)
print(r.status_code)
show(r.json())
token = r.json().get("access_token")

In [8]:
auth_headers = {**HEADERS, "Authorization": f"Bearer {token}"}
r = requests.get(f"{SECURE_URL}/Patient/{patient_id}", headers=auth_headers)
print(r.status_code)
show(r.json())

NameError: name 'token' is not defined